# Sample protein-coding GTEx Lung loci into a manifest

This notebook finds all local GTEx Lung allpairs parquet files, samples 20 protein-coding genes from each chromosome file, maps them to gene names using GENCODE, and writes a batch-ready manifest CSV.


In [ ]:
import gzip
import re
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


def find_project_root(start: Path | None = None) -> Path:
    candidates = []
    current = (start or Path.cwd()).resolve()
    candidates.extend([current, *current.parents])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate

    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.annotations import download_gtf_if_needed
from utils.paths import configure_runtime_env

PATHS = configure_runtime_env(PROJECT_ROOT)
CONFIG_DIR = PROJECT_ROOT / 'config'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_GENOME = 'hg38'
GTEX_TISSUE = 'Lung'
GENES_PER_CHROM = 100
RANDOM_SEED = 20260406
OUTPUT_MANIFEST_PATH = CONFIG_DIR / 'loci_manifest_sample_100_per_chrom.csv'

GTEX_PARQUET_PATHS = sorted(PATHS.gtex.glob(f'*{GTEX_TISSUE}*.allpairs.chr*.parquet'))
if not GTEX_PARQUET_PATHS:
    raise FileNotFoundError(f'No GTEx parquet files matched tissue {GTEX_TISSUE!r} in {PATHS.gtex}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Found {len(GTEX_PARQUET_PATHS)} GTEx parquet files for tissue {GTEX_TISSUE}:')
for parquet_path in GTEX_PARQUET_PATHS:
    print(f'  - {parquet_path.name}')
print(f'Output manifest: {OUTPUT_MANIFEST_PATH}')


In [ ]:
per_chrom_gene_tables = []
for parquet_path in GTEX_PARQUET_PATHS:
    gene_id_table = pq.read_table(parquet_path, columns=['gene_id'])
    gene_ids = (
        pd.Series(gene_id_table.column('gene_id').to_pylist(), name='gene_id')
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
    chrom_match = re.search(r'(chr[^.]+)\.parquet$', parquet_path.name)
    if chrom_match is None:
        raise ValueError(f'Could not parse chromosome from parquet filename: {parquet_path.name}')
    gtex_chrom = chrom_match.group(1)
    per_chrom_gene_tables.append(
        pd.DataFrame(
            {
                'gene_id': gene_ids,
                'gtex_chrom': gtex_chrom,
                'parquet_path': str(parquet_path),
            }
        )
    )

all_genes_df = pd.concat(per_chrom_gene_tables, ignore_index=True)
all_genes_df['gene_id_base'] = all_genes_df['gene_id'].str.replace(r'\.\d+$', '', regex=True)

chrom_summary_df = (
    all_genes_df.groupby('gtex_chrom', as_index=False)
    .agg(unique_gene_ids=('gene_id', 'nunique'))
    .sort_values('gtex_chrom')
    .reset_index(drop=True)
)
print('Unique gene_ids per chromosome before gene_type filtering:')
display(chrom_summary_df)
all_genes_df.head()


In [ ]:
gtf_path = download_gtf_if_needed(PATHS.gtf_cache, genome=REFERENCE_GENOME)
all_gene_id_bases = set(all_genes_df['gene_id_base'])

gene_records = []
with gzip.open(gtf_path, 'rt') as handle:
    for line in handle:
        if not line or line.startswith('#'):
            continue
        fields = line.strip().split('	')
        if len(fields) < 9 or fields[2] != 'gene':
            continue

        attributes = fields[8]
        attr_dict = {}
        for attr in attributes.split(';'):
            attr = attr.strip()
            if not attr:
                continue
            key_value = attr.split(' ', 1)
            if len(key_value) == 2:
                key, value = key_value
                attr_dict[key] = value.strip('"')

        gene_id = attr_dict.get('gene_id', '')
        gene_id_base = gene_id.split('.', 1)[0]
        if gene_id_base not in all_gene_id_bases:
            continue

        gene_records.append(
            {
                'gene_id_gtf': gene_id,
                'gene_id_base_gtf': gene_id_base,
                'gene_name': attr_dict.get('gene_name', ''),
                'gene_type': attr_dict.get('gene_type', ''),
            }
        )

gtf_gene_df = pd.DataFrame(gene_records).drop_duplicates(subset=['gene_id_gtf', 'gene_name']).reset_index(drop=True)
protein_coding_gene_bases = set(
    gtf_gene_df.loc[gtf_gene_df['gene_type'] == 'protein_coding', 'gene_id_base_gtf']
)
protein_coding_genes_df = all_genes_df[
    all_genes_df['gene_id_base'].isin(protein_coding_gene_bases)
].copy()

protein_coding_summary_df = (
    protein_coding_genes_df.groupby('gtex_chrom', as_index=False)
    .agg(protein_coding_gene_ids=('gene_id', 'nunique'))
    .sort_values('gtex_chrom')
    .reset_index(drop=True)
)
print('Protein-coding gene_ids per chromosome:')
display(protein_coding_summary_df)

if (protein_coding_summary_df['protein_coding_gene_ids'] < GENES_PER_CHROM).any():
    insufficient = protein_coding_summary_df[
        protein_coding_summary_df['protein_coding_gene_ids'] < GENES_PER_CHROM
    ]
    raise ValueError(
        'At least one chromosome does not have enough protein-coding genes for sampling:\n'
        + insufficient.to_string(index=False)
    )


In [ ]:
sampled_tables = []
for chrom_idx, chrom in enumerate(sorted(protein_coding_genes_df['gtex_chrom'].unique())):
    chrom_df = (
        protein_coding_genes_df[protein_coding_genes_df['gtex_chrom'] == chrom]
        .drop_duplicates(subset=['gene_id'])
        .copy()
    )
    sampled_chrom_df = chrom_df.sample(n=GENES_PER_CHROM, random_state=RANDOM_SEED + chrom_idx).copy()
    sampled_tables.append(sampled_chrom_df)

sampled_df = pd.concat(sampled_tables, ignore_index=True).sort_values(['gtex_chrom', 'gene_id']).reset_index(drop=True)

manifest_df = sampled_df.merge(
    gtf_gene_df[gtf_gene_df['gene_type'] == 'protein_coding'][['gene_id_gtf', 'gene_id_base_gtf', 'gene_name', 'gene_type']],
    left_on='gene_id_base',
    right_on='gene_id_base_gtf',
    how='left',
)

manifest_df['gene_name'] = manifest_df['gene_name'].fillna(manifest_df['gene_id_base'])
manifest_df['gene_name_slug'] = (
    manifest_df['gene_name']
    .astype(str)
    .str.replace(r'[^A-Za-z0-9]+', '_', regex=True)
    .str.strip('_')
    .str.lower()
)
manifest_df.loc[manifest_df['gene_name_slug'] == '', 'gene_name_slug'] = manifest_df['gene_id_base'].str.lower()
manifest_df['chrom_slug'] = manifest_df['gtex_chrom'].str.lower()
manifest_df['locus_id'] = manifest_df['gene_name_slug'] + '_' + manifest_df['chrom_slug'] + '_' + GTEX_TISSUE.lower()
manifest_df['gene_name_rank'] = manifest_df.groupby('locus_id').cumcount() + 1
manifest_df.loc[
    manifest_df['gene_name_rank'] > 1,
    'locus_id'
] = (
    manifest_df.loc[manifest_df['gene_name_rank'] > 1, 'locus_id']
    + '_' + manifest_df.loc[manifest_df['gene_name_rank'] > 1, 'gene_name_rank'].astype(str)
)

manifest_df = manifest_df.assign(
    gtex_tissue=GTEX_TISSUE,
    reference_genome='hg38',
    maf_min=0.01,
    maf_max=0.99,
    min_sample_size=50,
    alphagenome_sequence_length='1MB',
    alphagenome_target_data_source='gtex',
    alphagenome_target_gtex_tissue=GTEX_TISSUE,
    alphagenome_batch_size=100,
    alphagenome_max_workers=8,
    alphagenome_retry_wait_seconds=5,
    enabled=True,
)

manifest_df = manifest_df[
    [
        'locus_id',
        'gene_name',
        'gene_id',
        'gene_type',
        'gtex_tissue',
        'gtex_chrom',
        'reference_genome',
        'maf_min',
        'maf_max',
        'min_sample_size',
        'alphagenome_sequence_length',
        'alphagenome_target_data_source',
        'alphagenome_target_gtex_tissue',
        'alphagenome_batch_size',
        'alphagenome_max_workers',
        'alphagenome_retry_wait_seconds',
        'enabled',
    ]
].copy()

if manifest_df['locus_id'].duplicated().any():
    duplicated_ids = manifest_df.loc[manifest_df['locus_id'].duplicated(), 'locus_id'].tolist()
    raise ValueError(f'Duplicate locus_id values generated: {duplicated_ids[:10]}')

if not manifest_df['gene_type'].eq('protein_coding').all():
    raise ValueError('Non-protein-coding genes leaked into the sampled manifest.')

manifest_df.to_csv(OUTPUT_MANIFEST_PATH, index=False)
print(f'Wrote sampled manifest: {OUTPUT_MANIFEST_PATH}')
print(f'Total sampled loci: {len(manifest_df)}')
print('Sample counts per chromosome:')
display(manifest_df.groupby('gtex_chrom', as_index=False).size())
manifest_df.head(20)
